<img src="https://raw.githubusercontent.com/drdave-teaching/OPIM5641-notebooks/main/_banners/opim5641_banner.svg" width="100%" alt="OPIM 5641 banner"/>

**Dr. Dave Wanik - Operations and Information Management - University of Connecticut**

---

# Studio 3.3 — The Diet Problem: Feed a Football Team at McDonald's

The team bus pulls into McDonald's. **53 players**, one day of meals, and the athletic department is
paying. What's the cheapest order that keeps everyone fueled?

It's Stigler's diet again, with **three changes**:

| | Stigler | McDonald's |
|---|---|---|
| **Decision variable** | dollars per day on each food | **how many of each menu item** to order for the whole team |
| **Objective** | add up the dollars | add up **price × how many** |
| **Rules** | every nutrient has a minimum (`>=`) | some are minimums (`>=`), some are **maximums** (`<=`), and they are **per player**, so multiply by the team size |

Plus one new rule for variety: **nobody gets more than 4 of the same item.**

**How this notebook works.** All of the code is already here. You do not have to write a model
from scratch. Your job is to **run it, read it, and change it**: each experiment has a clearly marked
cell where you plug in a scenario, re-run, and see what happens.

**About the data (it is real):**
- **Nutrition** comes from **MenuStat**, the New York City Health Department's database of chain
  restaurant nutrition (2018 edition).
- **Prices** are McDonald's menu prices as listed on fastfoodmenuprices.com (updated July 11, 2026).
  Prices vary by location, so yours may be a little different.
- The menu has the items that appear in both sources.

**Before you solve it, guess:** what's the cheapest bill per player? Write it down.

## Import modules (download Pyomo)

Run this first, every time. It installs Pyomo and the `cbc` solver in Colab.

In [ ]:
# import modules

# this makes beautiful plots using pylab (matplotlib)
%matplotlib inline
from pylab import * # * means import ALL NAME SPACES
import pandas as pd

# useful modules for downloading pyomo onto Colab
import shutil
import sys
import os.path

# install pyomo if it doesn't exist
if not shutil.which("pyomo"):
    !pip install -q pyomo
    assert(shutil.which("pyomo"))

# install the 'cbc' solver if it doesn't exist
if not (shutil.which("cbc") or os.path.isfile("cbc")):
    if "google.colab" in sys.modules:
        !apt-get install -y -qq coinor-cbc
    else:
        try:
            !conda install -c conda-forge coincbc
        except:
            pass

# make sure 'cbc' is the solver that we will invoke later on
assert(shutil.which("cbc") or os.path.isfile("cbc"))

# import ALL NAMESPACES (variables) from pyomo
from pyomo.environ import *

# Part 1 · The data

Two CSV files: the **menu** (one row per item) and the **rules** (per player, per day).

`menu_file` is the only thing that says "McDonald's" in this whole notebook. Remember that: your
assignment will be to swap in another restaurant.

In [ ]:
# read the data
data_folder = 'https://raw.githubusercontent.com/drdave-teaching/OPIM5641-notebooks/main/studio3/menus/'
menu_file = 'mcdonalds_menu.csv'
rules_file = 'team_rules.csv'

menu_df = pd.read_csv(data_folder + menu_file)
rules_df = pd.read_csv(data_folder + rules_file)

menu_df

Each row is one item as you would order it: its price, and what is in **one** of it (calories,
grams of protein, carbs, fiber, sugar, fat and saturated fat, and milligrams of sodium).

## The rules, and how they compare with a food label

The side of a box of Triscuits shows **% Daily Value**. Those percentages are based on the FDA's
Daily Values for an average adult eating **2,000 calories** a day. A college football player is not
an average adult: a big lineman can burn twice that, needs a lot more protein to build muscle, and
sweats out a lot of salt at practice. So our rules are bigger than the label's.

In [ ]:
# the rules, per player per day - notice calories has BOTH a min and a max
rules_df

| | Food label Daily Value (average adult, 2,000 calories) | Our rule for one football player |
|---|---|---|
| Calories | 2,000 (the label's reference day) | at least 4,000, at most 5,500 |
| Protein | 50 g | at least 150 g |
| Fiber | 28 g | at least 25 g |
| Fat | 78 g | at most 180 g |
| Saturated fat | 20 g | at most 60 g |
| Sugar | 50 g of *added* sugar | at most 150 g of *total* sugar (the menu only lists total) |
| Sodium | 2,300 mg | at most 7,000 mg |

**These rules are our assumptions** for a big college football player on a training day, chosen to make
a good class example. A team dietitian would set real ones. Notice there is **no rule for carbs**.

## Soft-code the data into dictionaries

Same pattern as Flair's and Stigler. Five dictionaries:

- `price[item]` - what one of that item costs.
- `nutrition[item][nutrient]` - what is in one of that item. A dictionary of dictionaries.
- `label[nutrient]` - a readable name for printing, like `'Sodium (mg)'` for `'sodium'`.
- `player_min[nutrient]` - the **minimums** from the rules file (the rows that say `min`).
- `player_max[nutrient]` - the **maximums** from the rules file (the rows that say `max`).

Calories shows up in **both** `player_min` and `player_max`: at least 4,000, at most 5,500.

In [ ]:
# ---------- 1. THE DATA ----------
# the menu items - every column after 'price' is a nutrient
items = list(menu_df['item'])
nutrient_columns = list(menu_df.columns[3:])

# price[item] and nutrition[item][nutrient] (for ONE of that item)
price = {}
nutrition = {}
for i in range(len(menu_df)):
    item = menu_df.loc[i, 'item']
    price[item] = float(menu_df.loc[i, 'price'])
    nutrition[item] = {}
    for nutrient in nutrient_columns:
        nutrition[item][nutrient] = float(menu_df.loc[i, nutrient])

# a readable label for each nutrient (the rules file has nicer ones)
label = {}
for nutrient in nutrient_columns:
    label[nutrient] = nutrient
label['carbs'] = 'Carbohydrates (g)'

# the rules per player: one dictionary of minimums, one of maximums
player_min = {}
player_max = {}
for i in range(len(rules_df)):
    nutrient = rules_df.loc[i, 'nutrient']
    label[nutrient] = rules_df.loc[i, 'label']
    if rules_df.loc[i, 'rule'] == 'min':
        player_min[nutrient] = int(rules_df.loc[i, 'per_player'])
    else:
        player_max[nutrient] = int(rules_df.loc[i, 'per_player'])

Let's look inside each dictionary.

In [ ]:
# what is in each dictionary?
first_item = items[0]

print("price -", len(price), "items. For example:")
print("   price['" + first_item + "'] =", price[first_item])
print()
print("nutrition - what is in ONE of each item. For example:")
print("   nutrition['" + first_item + "'] =", nutrition[first_item])
print()
print("label - a readable name for each nutrient:")
print("  ", label)
print()
print("player_min - the minimums per player:", player_min)
print("player_max - the maximums per player:", player_max)

**The scoreboard.** An empty dictionary. Every time you run the solve cell, it adds one entry with
`scoreboard[scenario] = cost per player` (a new key adds a new entry). Run this cell **once**: running it
again wipes the scoreboard clean.

In [ ]:
# an empty scoreboard: scenario name -> cost per player
scoreboard = {}

# Part 2 · The scenario

Leave this cell alone for the first run. Two of the settings need a word of explanation:

- `banned` is a list of items **nobody** will eat. The model gets one extra rule per banned item:
  "order exactly 0 of it".
- `must_have` says which items **every** player gets, and how many each. The model gets one extra rule
  per item: "order at least that many × 53".

In [ ]:
# ---------- 2. THE SCENARIO (change things here) ----------
scenario = "1 base"
team_size = 53              # players on the roster
max_of_one_item = 4         # nobody gets more than 4 of the same item
low = dict(player_min)      # a fresh copy of the minimums per player
high = dict(player_max)     # a fresh copy of the maximums per player
banned = []                 # items nobody will eat, for example ['Filet-O-Fish']
must_have = {}              # items everybody gets, and how many each, for example {'Big Mac' : 1}

# Scenario 2: no variety rule (remove the # from the next two lines)
# scenario = "2 no variety rule"
# max_of_one_item = 100

# Scenario 3: coach's order - everybody gets a Big Mac and a medium fries
# scenario = "3 coach: Big Mac + fries"
# must_have = {'Big Mac' : 1, 'Medium French Fries' : 1}

# Scenario 4: the team doctor cuts the sodium limit to 5,000 mg
# scenario = "4 sodium 5,000"
# high['sodium'] = 5000

# Scenario 5: break it - the FDA says 2,300 mg of sodium a day
# scenario = "5 sodium 2,300"
# high['sodium'] = 2300

# Scenario 6: your own idea. Some things you can change:
#   team_size = 11                      a soccer team instead
#   low['protein'] = 200                linemen need more protein
#   high['sugar'] = 50                  the nutritionist hates sugar
#   banned = ['Filet-O-Fish']           nobody eats the fish
#   must_have = {'Apple Slices' : 2}    everybody gets two bags of apple slices


# Part 3 · The model, one piece at a time

**3a · The decision variables.** One variable per menu item: how many of it to order for the whole team.

We let these be **any number, not just whole numbers**. The solver might say "order 83.7 McChickens".
Imagine we can cut a sandwich with a knife and split it. Forcing whole numbers is **integer
programming**, which is coming in Module 5.

In [ ]:
# ---------- 3a. the decision variables ----------
print("The menu items (one decision variable each):")
for item in items:
    print("  ", item)

model = ConcreteModel()

# how many of each item to order for the whole team
model.x = Var(items, domain=NonNegativeReals)

print()
print("Number of decision variables =", len(model.x))

**3b · The objective: the total bill.** The same `+=` triangle as before, but now each row is
**price × how many**:

```
start               obj_expr = 0
after item 1        obj_expr = 4.29*x[Double Cheeseburger]
after item 2        obj_expr = 4.29*x[Double Cheeseburger] + 6.19*x[Big Mac]
   ...
after item 23       obj_expr = 4.29*x[Double Cheeseburger] + 6.19*x[Big Mac] + ... + 1.39*x[Apple Slices]
```

In [ ]:
# ---------- 3b. the objective ----------
obj_expr = 0
k = 1
for item in items:
    obj_expr += price[item] * model.x[item]
    if k <= 3:
        print("after item", k, ":", obj_expr)
    k += 1

print("   ...")
print("after item", len(items), ": a sum of", len(items), "terms")

model.cost = Objective(
                      expr = obj_expr,
                      sense = minimize)

**3c · The constraints.** Five loops, one for each kind of rule. They all go into the same
`ConstraintList`, which numbers them 1, 2, 3, ... in the order they are added. The cell prints every
constraint as it is added.

1. **Minimums** (from `low`): the team's total of a nutrient must be at least the per-player minimum × 53.
2. **Maximums** (from `high`): the team's total must be at most the per-player maximum × 53.
3. **Variety**: no item more than 4 × 53 times.
4. **Banned items** (scenario): exactly 0. Empty in the base scenario, so this loop adds nothing.
5. **Must-have items** (scenario): at least that many × 53. Also empty in the base scenario.

In [ ]:
# ---------- 3c. the constraints ----------
model.constraints = ConstraintList()

# 1. the minimums (multiply by the team size!)
print("1. MINIMUMS")
for nutrient in low:
    nutrient_expr = 0
    for item in items:
        nutrient_expr += nutrition[item][nutrient] * model.x[item]
    model.constraints.add(nutrient_expr >= low[nutrient] * team_size)
    print("   constraint", len(model.constraints), ":", label[nutrient], "for the team >=", low[nutrient], "x", team_size, "=", low[nutrient] * team_size)

# 2. the maximums
print("2. MAXIMUMS")
for nutrient in high:
    nutrient_expr = 0
    for item in items:
        nutrient_expr += nutrition[item][nutrient] * model.x[item]
    model.constraints.add(nutrient_expr <= high[nutrient] * team_size)
    print("   constraint", len(model.constraints), ":", label[nutrient], "for the team <=", high[nutrient], "x", team_size, "=", high[nutrient] * team_size)

# 3. variety: no more than max_of_one_item of any item per player
print("3. VARIETY")
first = len(model.constraints) + 1
for item in items:
    model.constraints.add(model.x[item] <= max_of_one_item * team_size)
print("   constraints", first, "to", len(model.constraints), ": each item <=", max_of_one_item, "x", team_size, "=", max_of_one_item * team_size)

# 4. scenario rule: banned items get exactly 0
print("4. BANNED ITEMS")
for item in banned:
    model.constraints.add(model.x[item] == 0)
    print("   constraint", len(model.constraints), ":", item, "= 0")
if len(banned) == 0:
    print("   (none in this scenario)")

# 5. scenario rule: items everybody gets
print("5. MUST-HAVE ITEMS")
for item in must_have:
    model.constraints.add(model.x[item] >= must_have[item] * team_size)
    print("   constraint", len(model.constraints), ":", item, ">=", must_have[item], "x", team_size)
if len(must_have) == 0:
    print("   (none in this scenario)")

# ---------- YOUR OWN CONSTRAINT (optional) ----------
# You can also write a constraint yourself. For example, at least as many fries as cheeseburgers:
# model.constraints.add(model.x['Medium French Fries'] >= model.x['Cheeseburger'])

print()
print("Total number of constraints =", len(model.constraints))

**3d · Solve it and see the order.** This cell solves the model, prints the order (every item the team
orders at least half of), checks the nutrition per player against each rule, flags the **binding** ones,
draws the order, and writes the result on the scoreboard.

In [ ]:
# ---------- 3d. solve and report (you never change this cell) ----------
results = SolverFactory('cbc', executable='/usr/bin/cbc').solve(model)

print("SCENARIO:", scenario)
if results.solver.termination_condition == TerminationCondition.optimal:
    per_player = round(model.cost() / team_size, 2)
    print("Total bill = $", round(model.cost(), 2))
    print("Per player = $", per_player)

    print()
    print("The order:")
    names = []
    counts = []
    for item in items:
        if model.x[item]() >= 0.5:          # the team orders at least half of one
            print("  ", item, ":", round(model.x[item](), 1), "for the team (", round(model.x[item]() / team_size, 2), "per player )")
            names.append(item)
            counts.append(model.x[item]() / team_size)

    print()
    print("Per player per day:")
    for nutrient in nutrient_columns:
        total = 0
        for item in items:
            total += nutrition[item][nutrient] * model.x[item]()
        amount = total / team_size
        note = ""
        if nutrient in low:
            note += " min " + str(low[nutrient])
            if abs(amount - low[nutrient]) < 0.01:
                note += "  <-- BINDING"
        if nutrient in high:
            note += " max " + str(high[nutrient])
            if abs(amount - high[nutrient]) < 0.01:
                note += "  <-- BINDING"
        if note == "":
            note = " no rule"
        print("  ", label[nutrient], ":", round(amount, 1), "|" + note)

    # add this scenario to the scoreboard dictionary: scoreboard[key] = value
    scoreboard[scenario] = per_player

    # a picture of the order
    barh(names, counts, color='darkorange')
    xlabel('how many per player')
    title(scenario + ':  $%.2f per player' % per_player)
    tight_layout()
    savefig('fig_team_order.png')
    show()
else:
    print("INFEASIBLE! No order can satisfy every rule at the same time.")
    print("Loosen a rule and try again.")

How close was your guess?

**About \$36.73 per player**, or \$1,946.51 for the team: lots of cheeseburgers and hamburgers, plus
McChickens, fries and vanilla shakes. The **binding** rules (calories, fiber, sugar) are what is
driving the bill. Protein and sodium have room to spare, and there is no rule on carbs at all.

### Try it: play with the scenarios

For each scenario: remove the `#` from that scenario's lines in **THE SCENARIO** cell, then click on the
scenario cell and choose **Runtime → Run cell and below** (Ctrl+F10). Put the `#` back before you try
the next one.

| Scenario | What changes | Cost per player | What changed in the order? |
|---|---|---|---|
| 1 base | nothing | | |
| 2 | no variety rule | | |
| 3 | coach: everybody gets a Big Mac and fries | | |
| 4 | sodium limit down to 5,000 mg | | |
| 5 | sodium limit down to 2,300 mg (the FDA's number) | | |
| 6 | your own idea | | |

Then try writing **one constraint of your own** at the bottom of the constraints cell (3c). The example
there says "at least as many fries as cheeseburgers". What rule would your team want?

In [ ]:
# the scoreboard so far
print(scoreboard)

## Your deliverable: the team order card

When you've played your scenarios, run the cell below to draw your scoreboard, then answer the three
questions underneath it. That's what you turn in tonight.

In [ ]:
# draw the scoreboard
scenario_names = []
dollars = []
for name in scoreboard:
    scenario_names.append(name)
    dollars.append(scoreboard[name])

barh(scenario_names, dollars, color='steelblue')
xlabel('$ per player')
title('Feed the team: what did each rule cost?')
gca().invert_yaxis()
tight_layout()
savefig('fig_scoreboard.png')
show()

**Double-click this cell and answer:**

1. **Scenario 2:** what did the cheapest order look like with no variety rule, and how much did
   variety cost per player?

   *your answer*

2. **Scenarios 4 and 5:** what did cutting the sodium to 5,000 mg cost per player? What happened at
   2,300 mg, and what does that tell you about a day of McDonald's?

   *your answer*

3. **Scenario 6:** what did you change, and what did it cost the team? Was it worth it?

   *your answer*

## Bottom line

- Stigler's diet and the McDonald's order are **the same model with different CSV files.**
- A minimum is a `>=` constraint, a maximum is a `<=` constraint. Same loop, different sign.
- **Binding** rules drive the bill. **Infeasible** means the rules contradict each other: loosen a
  rule, don't fix the code.
- Swap `menu_file` and the same notebook works for any restaurant. That's your assignment.

**Before you move on:** File → Save a copy in GitHub.